# Μῆτις (Metis) — Train on Google Colab GPU

Train your Metis model on Colab's free GPU and save the checkpoints to your **Google Drive**.

**8 simple steps — just run each cell top to bottom:**
1. Mount Google Drive
2. Clone the GitHub repo (`iamasrakib/Metis`) — it's public, no login needed
3. Install dependencies
4. Generate the West Bengal dataset
5. Link the checkpoint folder into Google Drive (checkpoints save there as it trains)
6. Train on the GPU — `train_westbengal_100m.py` (100M params)
7. Confirm the checkpoints landed in Drive
8. (Optional) Generate a sample from the trained model

---
## Step 1 — Mount Google Drive

Your checkpoints will be stored here: `MyDrive/Metis/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Step 2 — Clone the Metis repo

The repo is public — the next cell just clones it, no login needed.

In [ ]:
import os

REPO = "https://github.com/iamasrakib/Metis.git"
METIS_DIR = "/content/Metis"

if os.path.isdir(METIS_DIR):
    # Re-run: sync to the latest code (discards local edits like max_iters).
    os.system(f"git -C {METIS_DIR} fetch origin")
    ret = os.system(f"git -C {METIS_DIR} reset --hard origin/main")
else:
    ret = os.system(f"git clone {REPO} {METIS_DIR}")
if ret != 0:
    raise RuntimeError("git failed - is the repo public?")

%cd /content/Metis

---
## Step 3 — Install dependencies

PyTorch (with CUDA) is already installed on Colab — this installs the small set of extra packages Metis needs for training.

In [ ]:
!pip install -q numpy tqdm tiktoken tokenizers

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("No GPU found! Enable one: Runtime > Change runtime type > T4 GPU")

---
## Step 4 — Generate the dataset

The generator scripts are self-contained (the text is baked in), so nothing to download. Default: **West Bengal**. Switch by uncommenting another line.

In [ ]:
!python data/generate_westbengal_dataset.py
# !python data/generate_cow_dataset.py      # cow knowledge corpus
# !python data/generate_siraj_dataset.py    # Siraj-ud-Daulah history

---
## Step 5 — Link checkpoints straight into Google Drive

Creates `MyDrive/Metis/checkpoints_westbengal_100m/` and symlinks it to where training writes. Every checkpoint is **saved directly to Drive as training runs** — even if this Colab session disconnects mid-training, nothing is lost.

In [ ]:
import os

CKPT_DIR   = "checkpoints_westbengal_100m"       # matches train_westbengal_100m.py
DRIVE_BASE = "/content/drive/MyDrive/Metis"

os.makedirs(DRIVE_BASE, exist_ok=True)
drive_ckpt = os.path.join(DRIVE_BASE, CKPT_DIR)
os.makedirs(drive_ckpt, exist_ok=True)

local_link = os.path.abspath(CKPT_DIR)
if not os.path.lexists(local_link):
    os.symlink(drive_ckpt, local_link)
    print(f"Linked  {local_link} -> {drive_ckpt}")
else:
    print("Already linked:", local_link)

print("\nTrain a different model? Change CKPT_DIR to match the script:")
print("  train_westbengal_100m.py  -> checkpoints_westbengal_100m")
print("  train_cow.py              -> checkpoints_cow")
print("  train_westbengal_small.py -> checkpoints_westbengal_small")

---
## Step 6 — Train on the GPU

The 100M config ships with `max_iters=30` (just a smoke test). Set how many steps you want — for this 36K-char corpus **500–2000** works well. Training prints live progress.

In [ ]:
import re

MAX_ITERS = 1000   # <-- how many optimizer steps to train

src = "train_westbengal_100m.py"
with open(src, encoding="utf-8") as f:
    code = f.read()
code = re.sub(r"max_iters=\d+", f"max_iters={MAX_ITERS}", code)
with open(src, "w", encoding="utf-8") as f:
    f.write(code)
print(f"max_iters -> {MAX_ITERS}")

# Train on the Colab GPU. Checkpoints stream into Google Drive (Step 5).
!python train_westbengal_100m.py

---
## Step 7 — Confirm the checkpoints are in Drive

In [ ]:
import os

ckpt_dir = "/content/drive/MyDrive/Metis/checkpoints_westbengal_100m"
for name in sorted(os.listdir(ckpt_dir)):
    path = os.path.join(ckpt_dir, name)
    size = os.path.getsize(path)
    line = f"  {name:<28} {size/1e6:6.2f} MB" if size >= 1e6 else f"  {name:<28} {size:>7,} B"
    print(line)

print("\nSaved in Google Drive: MyDrive/Metis/checkpoints_westbengal_100m/")
print("Chat with it on your PC later:  metis chat --checkpoint-dir checkpoints_westbengal_100m")

---
## Step 8 — (Optional) Test the model here

Generates a short continuation on the GPU from the freshly trained weights.

In [ ]:
!python generate.py --prompt "West Bengal is" --max-tokens 150 --checkpoint-dir checkpoints_westbengal_100m

# Step 9 — Optional: Combined model (West Bengal + Bihar)

Your West Bengal model already works. This optional path builds a **bigger model that knows BOTH states**:
1. Generate the Bihar dataset (about 38K characters).
2. Train one new 100M model on both corpora at once — the train script merges them automatically into `data/combined.txt` (about 75K chars).
3. Checkpoints save to `MyDrive/Metis/checkpoints_combined_100m/` (same Drive workflow as before).

In [ ]:
# Generate the Bihar dataset. The combined train script merges it with
# West Bengal automatically, so you don't have to combine anything by hand.
!python data/generate_bihar_dataset.py

## Step 10 — Train the combined model

Link the checkpoints folder into Google Drive first (so training saves straight to Drive), then run the train cell.

In [ ]:
import os

CKPT_DIR   = "checkpoints_combined_100m"       # matches train_combined_100m.py
DRIVE_BASE = "/content/drive/MyDrive/Metis"

os.makedirs(DRIVE_BASE, exist_ok=True)
drive_ckpt = os.path.join(DRIVE_BASE, CKPT_DIR)
os.makedirs(drive_ckpt, exist_ok=True)

local_link = os.path.abspath(CKPT_DIR)
if not os.path.lexists(local_link):
    os.symlink(drive_ckpt, local_link)
    print(f"Linked  {local_link}
       -> {drive_ckpt}")
else:
    print("Already linked:", local_link)

## Step 11 — Train

Set how many steps you want — for this ~75K-character corpus, **500–2000** works well. Then run the cell; training streams progress live.

In [ ]:
import re

MAX_ITERS = 1000   # <-- how many optimizer steps to train the combined model

src = "train_combined_100m.py"
with open(src, encoding="utf-8") as f:
    code = f.read()
code = re.sub(r"max_iters=\d+", f"max_iters={MAX_ITERS}", code)
with open(src, "w", encoding="utf-8") as f:
    f.write(code)
print(f"max_iters -> {MAX_ITERS}")

# Train on the combined West Bengal + Bihar corpus.
!python train_combined_100m.py

## Step 12 — Test the combined model

Generate a sample continuation from the freshly trained combined weights.

In [ ]:
!python generate.py --prompt "Bihar is" --max-tokens 150 --checkpoint-dir checkpoints_combined_100m

# Step 13 — Distill from an API (train forever)

Instead of a fixed text file, this trains Metis **forever** on text written by
a frontier teacher model (ChatGPT / DeepSeek / Claude) reached through your
custom **omniroute** API. The loop never stops on its own — the teacher keeps
writing, Metis keeps learning, and checkpoints stream into Drive.

⚠️ **Cost warning:** an infinite loop calls your API continuously. Control
spend with `--max-tokens`, `--min-sleep`, and `--budget-tokens`.

**You need three things from your gateway:**
- **Base URL** — must be a *public* URL. Colab runs in Google's cloud and
  cannot reach `localhost`, so expose your PC's gateway with a tunnel first,
  e.g. `cloudflared tunnel --url http://localhost:20128`, then use the
  `https://<random>.trycloudflare.com/v1` URL it prints.
- **API key** — your omniroute key
- **Model name** — e.g. `nvidia/deepseek-ai/deepseek-v4-flash`

Credentials are stored in **Colab Secrets** (left sidebar → key icon) — never
pasted into code, never committed to the repo.

**Stop anytime** → press `Ctrl+C` in the run cell (it saves first), or create a
file named `STOP` inside the checkpoint dir. **Re-run the run cell later and it
resumes from where it stopped — no setup.** You can close Colab and come back
to resume.

In [ ]:
import os
from google.colab import userdata

# Teacher credentials come from Colab Secrets (left sidebar -> key icon), so
# nothing sensitive is ever committed to the repo. Add these secrets BEFORE
# running this cell:
#   TEACHER_URL       your gateway's PUBLIC base URL, ending in /v1
#                     e.g. https://<random>.trycloudflare.com/v1
#   TEACHER_MODEL     e.g. nvidia/meta/llama-3.2-3b-instruct
#   TEACHER_API_KEY   any non-empty string (your local gateway does not check it)
def _secret(name, default=""):
    try:
        return userdata.get(name)
    except Exception:
        return default

os.environ["METIS_TEACHER_BASE_URL"] = _secret("TEACHER_URL")
os.environ["METIS_TEACHER_MODEL"]    = _secret("TEACHER_MODEL", "nvidia/meta/llama-3.2-3b-instruct")
os.environ["METIS_TEACHER_API_KEY"]  = _secret("TEACHER_API_KEY", "local-gateway")

if not os.environ["METIS_TEACHER_BASE_URL"]:
    print("Missing TEACHER_URL secret. Add it: left sidebar > Secrets > + New secret.")
    print("  TEACHER_URL = <your public tunnel URL ending in /v1>")
else:
    print("Teacher credentials loaded from Colab Secrets.")

In [ ]:
import os

KEY = os.environ.get("METIS_TEACHER_API_KEY", "")
if KEY:
    # One connectivity call to confirm the gateway before training forever.
    os.system("python -m metis.cli distill --preset tiny --test-teacher")
else:
    print("Set METIS_TEACHER_API_KEY in the cell above, then re-run this cell.")

In [ ]:
import os

DRIVE_BASE = "/content/drive/MyDrive/Metis"
CKPT_DIR   = "checkpoints_distill"

os.makedirs(DRIVE_BASE, exist_ok=True)
drive_ckpt = os.path.join(DRIVE_BASE, CKPT_DIR)
os.makedirs(drive_ckpt, exist_ok=True)
local_link = os.path.abspath(CKPT_DIR)
if not os.path.lexists(local_link):
    os.symlink(drive_ckpt, local_link)
    print("Linked ->", drive_ckpt)
else:
    print("Already linked:", local_link)

# Train FOREVER from the teacher API. Ctrl+C stops it (saves first); re-running
# this cell later resumes from Drive automatically. Edit the flags to taste.
# --min-sleep 30 keeps under the gateway's per-model rate-limit queue.
os.system("python -m metis.cli distill --checkpoint-dir checkpoints_distill "
          "--preset tiny --tokenizer cl100k_base --min-sleep 30")

**Tips**
- **Stop:** `Ctrl+C` in the run cell, or create a file `checkpoints_distill/STOP`.
- **Resume:** just re-run the run cell — it reloads the last checkpoint and
  tokenizer and continues; no setup.
- **What it writes about:** add `--topic "animals"` (or `--topic-file topics.txt`).
- **Limit spend:** add `--budget-tokens 100000` to stop after a token budget.
- **Char tokenizer:** pass `--tokenizer char --seed-data <corpus.txt>` so the
  char vocab is fit once and reused.